# Diagnosing an infeasible model with Gurobi's IIS Example

In this example a deliberately infeasible single region
energy system is modeled and then diagnosed using **Irreducible Inconsistent Subsystem (IIS)**.

When a scenario does not solve, the solver usually reports nothing but `infeasible`. That
single word gives no hint about which part of the model is impossible, so the common
reaction is trial and error: switching components off one by one until the model solves
again. On a large energy system model that is slow and error-prone.

Gurobi can do better. Given an infeasible model it computes an Irreducible Inconsistent
Subsystem (IIS): a minimal set of constraints and variable bounds which together cannot be
satisfied, and from which removing any single member would make the rest satisfiable. The
IIS usually points straight at the modeling mistake, for example a fixed demand that no
source can meet.

The workflow is structured as follows:
1. Import required packages
2. Create an infeasible energy system model instance
3. Optimize the model
4. Compute the IIS with Gurobi
5. Repair the model
6. Other solvers

# 1. Import required packages

In [ ]:
import fine as fn
import gurobipy as gp
import pandas as pd


%load_ext autoreload
%autoreload 2

# 2. Create an infeasible energy system model instance

One region, four hourly time steps and one commodity are considered. The only source is a
PV component whose capacity is capped at 1 GW, so it can supply at most 1 GWh per hour.
The sink fixes the electricity demand to 10 GW in every hour. The commodity balance
therefore cannot be closed at any time step, which makes the model infeasible by
construction and the diagnosis below reproducible.


In [ ]:
locations = {"Region"}
commodityUnitDict = {"electricity": r"GW$_{el}$"}
commodities = {"electricity"}
numberOfTimeSteps = 4
hoursPerTimeStep = 1

In [ ]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    numberOfTimeSteps=numberOfTimeSteps,
    commodityUnitsDict=commodityUnitDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
)

### PV with a capacity cap of 1 GW

In [ ]:
esM.add(
    fn.Source(
        esM=esM,
        name="PV",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=pd.Series(1.0, index=list(locations)),
        operationRateMax=pd.DataFrame(
            1.0, index=range(numberOfTimeSteps), columns=list(locations)
        ),
        investPerCapacity=0.0,
        opexPerCapacity=0.0,
        interestRate=0.0,
        economicLifetime=20,
    )
)

### Fixed demand of 10 GW which the source cannot meet

In [ ]:
esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=pd.DataFrame(
            10.0, index=range(numberOfTimeSteps), columns=list(locations)
        ),
    )
)

## 3. Optimize the model

In [ ]:
# The model is infeasible by construction, so the run ends without a solution.
# This is all the information the solver gives on its own: the problem is named,
# its cause is not.
esM.optimize()

## 4. Compute the IIS with Gurobi

**`computeIIS()`** computes the Irreducible Inconsistent Subsystem: a minimal set of
constraints (and variable bounds) that together cannot be satisfied, and from which
removing any single member would make the rest feasible. That minimal set is the conflict
core. It points straight at the modeling error instead of just reporting `infeasible`.

`esM.pyM` is the Pyomo model behind the energy system. It exists as soon as the
optimization problem has been declared, which `esM.optimize()` does before it hands the
problem to the solver. On a large model the failed solve can be skipped altogether by
calling `esM.declareOptimizationProblem()` instead.

For better readability, use the option `symbolic_solver_labels=True`. Without it, Pyomo
names the constraints `c1, c2, ...` and the resulting file is unreadable.

In [ ]:
# Write the infeasible model to an LP file
esM.pyM.write("infeasible_model.lp", io_options={"symbolic_solver_labels": True})

In [ ]:
gurobi_model = gp.read("infeasible_model.lp") # read LP file into Gurobi model
gurobi_model.setParam("OutputFlag", 0) # silence Gurobi's console logging

# computeIIS() computes the Irreducible Inconsistent Subsystem:
# the minimal set of mutually conflicting constraints.
# It returns nothing! instead it sets the boolean attribute IISConstr on every
# constraint (True if the constraint is part of the conflict core, False otherwise).
gurobi_model.computeIIS()


# save just the IIS (conflict core) as a file. Same LP format as .lp, but reduced to the conflicting constraints
gurobi_model.write("infeasible_model.ilp")

In [ ]:
conflicting_constraints = [c.ConstrName for c in gurobi_model.getConstrs() if c.IISConstr]
#print the conflicting constraints
print(f"{len(conflicting_constraints)} constraint(s) in the IIS:")
for name in conflicting_constraints:
    print(f"  {name}")

# The bounds belong to the IIS as well, and this is where the conflicting numbers sit:
# capacityMax and operationRateFix enter the problem as bounds, not as constraints.
print("\nBound(s) in the IIS:")
for v in gurobi_model.getVars():
    if v.IISLB:
        print(f"  {v.VarName} >= {v.LB}")
    if v.IISUB:
        print(f"  {v.VarName} <= {v.UB}")

## 4.1. Read the `.ilp` file


In [ ]:
print(open("infeasible_model.ilp").read())


An LP file splits a model
into two parts:

- **`Bounds`** — simple limits on a *single* variable on its own: `cap_PV <= 1`
  (capacity is at most 1) and `op_demand >= 10` (demand is at least 10). This is where the
  fixed numbers of the model live, since `operationRateFix` and `capacityMax` enter as bounds.
- **`Subject To`** — the rules that link *several* variables together:
  - **`op_PV = op_demand`** — the balance: PV must generate exactly what is consumed.
  - **`op_PV <= cap_PV`** — PV cannot generate more than its capacity.

Read together, the two parts spell out the contradiction: demand is 10, the balance forces
PV to generate 10, but PV can do at most 1. The bounds (10 and 1) are the numbers you would
change to fix it; the constraints are the rules that make them collide.

Note: 
- **It is *one* minimal conflict, not all of them.** If a model contains several
  independent errors, the IIS shows one of them. After repairing it the model can still be
  infeasible and the next call reveals the next conflict; diagnosis is iterative.

## 5. Repair the model

### Raise the bound the IIS named: `capacityMax` from 1 GW to 10 GW

The conflict core contained `cap_PV <= 1`, which collides with the fixed demand of 10.
Lifting exactly that bound resolves the contradiction and leaves the rest of the model as it
is — the capacity variable included.

In [ ]:
esM.removeComponent("PV")  # remove the old PV source

esM.add(
    fn.Source(
        esM=esM,
        name="PV",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=pd.Series(10.0, index=list(locations)),
        operationRateMax=pd.DataFrame(
            1.0, index=range(numberOfTimeSteps), columns=list(locations)
        ),
        investPerCapacity=0.0,
        opexPerCapacity=0.0,
        interestRate=0.0,
        economicLifetime=20,
    )
)

In [ ]:
# optimize the model again, now that it is feasible
esM.optimize()

## 6. Other solvers

IIS computation is a Gurobi feature. Among the open solvers, [HiGHS](https://highs.dev/)
has a similar, weaker version, [`getIis()`](https://ergo-code.github.io/HiGHS/dev/guide/advanced/#highs-iis).